# finance / portfolio / returns

Annual portfolio return to 2070 under a low, base and high scenario.

| Term | In This Notebook |
|---|---|
| **Intent** | Give one annual return per scenario to grow the portfolio to 2070. Placeholder: the base is the mean annual return and the low and high are one standard deviation either side |
| **Project Flavour** | `pyfunc`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten years of annual portfolio returns, synthetic until the wrangle series is loaded; `validation` five further years; `backtest` ten more, held back for acceptance; `golden` the exact scenarios that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/returns"
MODEL_NAME = "finance_portfolio_returns"
TARGETS = [{"kind": "served"}]
TRAINING = "portfolio_returns_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_returns_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_returns_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_returns_golden"
INPUTS = ["finance_portfolio_returns_scenario"]
OUTPUTS = ["finance_portfolio_returns_nominal_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, around a 6 percent mean and a 2.45 percent standard deviation, so no query runs.

In [ ]:
DEVIATIONS = [-4.0, -3.0, -2.0, -1.0, 0.0, 0.0, 1.0, 2.0, 3.0, 4.0]


def observed(deviations):
    return {INPUTS[0]: [1.0] * len(deviations), OUTPUTS[0]: [round(6.0 + deviation, 4) for deviation in deviations]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: {"year": list(range(2016, 2026)), "portfolio_return_pct": [round(6.0 + deviation, 4) for deviation in DEVIATIONS]})
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: observed([0.5, -0.5, 1.0, -1.0, 0.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: observed(DEVIATIONS[::-1]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the mean and the population standard deviation of the annual returns in `training` only (R6). The low and high scenarios are one standard deviation either side of the base.

In [ ]:
BASE = round(float(training_pl["portfolio_return_pct"].mean()), 2)
SPREAD = round(float(training_pl["portfolio_return_pct"].std(ddof=0)), 2)
PARAMS = {"low": round(BASE - SPREAD, 2), "base": BASE, "high": round(BASE + SPREAD, 2)}
print("fitted", PARAMS)
assert PARAMS == {"low": 3.55, "base": 6.0, "high": 8.45}, "the fit did not recover the mean and spread"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** write `model.py` from the fitted values, build the signature, run it through the library runtime and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from workbench.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = "finance_portfolio_returns_scenario"
OUTPUT = "finance_portfolio_returns_nominal_pct"
RETURNS = ({low!r}, {base!r}, {high!r})


class Returns(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [RETURNS[min(max(int(round(scenario)), 0), 2)] for scenario in model_input[INPUT]]}})


set_model(Returns())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input("finance_portfolio_returns_scenario", "none", "Scenario, 0 low, 1 base, 2 high."),
    ],
    [
        signature_output("finance_portfolio_returns_nominal_pct", "percent", "Annual portfolio return for the scenario, percent per year.", datatype="FP64"),
    ],
)
predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 3.0
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file from the shipped runtime, copy `model.py` and the evaluation into a project, verify it and replay the golden through the library runtime (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [0.0, 1.0, 2.0, 1.4, 1.6, -1.0, 3.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual portfolio returns, training split",
        [training_path, golden_path],
        "mean and standard deviation of annual returns per scenario",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the three scenarios and out of range ones",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)